# Pawnee National Grassland Land Swap
##  GBIF Animal Observations & Ecological Value Mapping
- **Objective:** 

For this notebook, the observations of prarie dogs and pronghorn (antelope) are pulled from [GBIF](https://www.gbif.org/) for each parcel within the Pawnee National Grassland boundary. A "ecological value" for these species will be generated for each parcel, and in the `parcel_matrix` notebook these ecological values will be appended to identify best land swaps.

- **Objective Goals:**
  - Query and download GBIF occurrence data for focal species within the Pawnee boundary  
  - Clean and spatially filter observations to align with parcel geometries  
  - Aggregate species observations at the parcel level  
  - Generate a standardized **ecological value score** based on species presence/abundance  
  - Prepare outputs for integration into land swap and fragmentation analysis workflows

- **Author:** Max Warnock
- **Code review and/or edits:** Kayleigh Ward
- **Date:** April 9, 2026
- **Last date of revision:** April 28, 2026

---

### 🛠️ Prerequisites & Setup

**Mandatory Libraries:**
- `pandas` – tabular data handling  
- `geopandas` – spatial data processing and clipping  
- `pygbif` - GBIF API access
- `requests` – API queries to GBIF  
- `holoviews`/ `hvplot` - interactive geospatial visualization

**Environment:**
- Conda environment (e.g., `pawnee-grasslands`) with geospatial dependencies installed  
- Internet connection required for querying GBIF data  

**Data Sources:**
- GBIF occurrence data (via API using `pygbif`)  
- Pawnee National Grassland parcel dataset  
- Western Pawnee master boundary shapefile (`pawnee_master_west.shp`)  

**Related Notebooks:**
- Must run the `01_boundaries` notebook prior to this notebook  
- This notebook relies on the generated Western Pawnee boundary shapefile:
  - `pawnee_master_west.shp`  
- Output is used in:
  - `parcel_matrix` notebook  

**Notes:**
- GBIF queries may return large datasets; filtering by bounding box and species is used to manage size  
- Spatial joins require consistent CRS between GBIF points and parcel geometries  

---

### 🏗️ Methodology

#### 1. Define Focal Species and Query GBIF  
- Identify target species (e.g., prairie dogs and pronghorn antelope)  
- Use `pygbif` to retrieve occurrence records within the Pawnee bounding region  

#### 2. Download and Load Occurrence Data  
- Submit GBIF download request and retrieve dataset  
- Load occurrence records into a pandas/GeoPandas workflow  

#### 3. Clean and Filter Observations  
- Remove records with missing or invalid coordinates  
- Convert to GeoDataFrame and ensure proper CRS alignment  
- Clip observations to the Western Pawnee boundary  

#### 4. Spatial Join with Parcels  
- Overlay occurrence points with parcel geometries  
- Assign species observations to individual parcels  

#### 5. Aggregate Observations by Parcel  
- Count or summarize species occurrences within each parcel  
- Standardize counts across species if needed  

#### 6. Generate Ecological Value Metric (not yet complete) 
- Create a parcel-level ecological value score based on species presence/abundance  
- Prepare output dataset for integration into land swap optimization  

---

### Reproducibility Notes
- GBIF data is dynamic and may change over time depending on new submissions or updates  
- Exact results may vary slightly depending on query date and filtering parameters  
- Ensure consistent CRS and boundary inputs for reproducible spatial joins  

---

### ⚡ Troubleshooting/Notes
* GBIF API requests may fail intermittently — retry queries if needed  
* Large downloads can take time; monitor GBIF download status if using asynchronous requests  
* CRS mismatches are a common source of spatial join errors — verify projections before analysis  
* Some species observations may fall outside expected ranges due to data quality issues — apply filters as needed  

# Libraries

In [1]:
### file paths, OS operations, utilities
import os
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

### data handling 
import pandas as pd
import geopandas as gpd

### web requests / data download
import requests

### geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

### GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species

# Primary Directory

In [2]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project


### Secondary Directories

In [3]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Western Pawnee
boundary_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')


### MASTER
### master boundary dir
master_bound_west = os.path.join(boundary_dir_west, 'master_boundary')

### master boundary shapefile
master_bound_west_path = os.path.join(master_bound_west, "pawnee_master_west.shp")

### master boundary convert to gdf
master_bound_west_gdf = gpd.read_file(master_bound_west_path)


### FEDERAL
### federal boundary dir
federal_bound_west = os.path.join(boundary_dir_west, 'federal_boundary')

### federal boundary shapefile 
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')

### federal boundary convert to gdf
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)


### STATE
### state boundary dir
state_bound_west = os.path.join(boundary_dir_west, 'state_boundary')

### state boundary shapefile 
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')

### state boundary convert to gdf
state_bound_west_gdf = gpd.read_file(state_bound_west_path)


### PARCEL
### parcel boundary dir
parcel_bound_west = os.path.join(boundary_dir_west, 'parcel_boundary')

### parcel boundary shapefile 
parcel_bound_west_path = os.path.join(parcel_bound_west, 'pawnee_parcel_west.shp')

### parcel boundary convert to gdf
parcel_bound_west_gdf = gpd.read_file(parcel_bound_west_path)

### GBIF Login

In [4]:
### reset credentials
reset_credentials = False

### make dictionary for GBIF username and pass
credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(getpass, 'GBIF password'),
    GBIF_EMAIL=(input, 'GBIF email'),
)

### loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():

    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

### Set directories for GBIF download

In [5]:
### set a directory for the GBIF data
gbif_dir = os.path.join(data_dir, 'gbif')
os.makedirs(gbif_dir, exist_ok=True)

In [6]:
### set a directory for the GBIF pronghorn data
gbif_pronghorn_dir = os.path.join(gbif_dir, 'gbif_pronghorn')
os.makedirs(gbif_pronghorn_dir, exist_ok=True)

In [7]:
### set a directory for the GBIF prairie dog data
gbif_prairie_dog_dir = os.path.join(gbif_dir, 'gbif_prairie_dog')
os.makedirs(gbif_prairie_dog_dir, exist_ok=True)

In [8]:
### set a directory for the clipped GBIF data
gbif_clipped_dir = os.path.join(gbif_dir, 'gbif_clipped')
os.makedirs(gbif_clipped_dir, exist_ok=True)

In [21]:
### output directory for figures
gbif_animal_fig_dir = os.path.join(repo_root, 'figures', 'animals')
os.makedirs(gbif_animal_fig_dir, exist_ok=True)

### Define download function

In [9]:
### create a function to download GBIF data
def download_gbif_species(species_name, gbif_dir):
    """
    download GBIF occurrence data for a species and return a dataframe

    Args:
    species_name (str): scientific name of the species
    gbif_dir (str): directory where GBIF data will be stored

    Returns:
    pandas.DataFrame: dataframe of GBIF occurrences
    str: path to the downloaded csv file
    int: GBIF species key
    """

    ### get species key
    species_info = species.name_backbone(species_name)
    species_key = int(
        species_info.get("usageKey") or species_info.get("usage", {}).get("key")
    )
    print(f"{species_name}: {species_key}")

    ### check for existing data
    csv_files = glob(os.path.join(gbif_dir, "*.csv"))

    if not csv_files:
        download_key = occ.download([
            f"speciesKey = {species_key}",
            "hasCoordinate = TRUE",
        ])[0]

        ### wait for download
        while True:
            status = occ.download_meta(download_key)["status"]
            if status == "SUCCEEDED":
                break
            if status in {"CANCELLED", "KILLED", "FAILED"}:
                raise RuntimeError(f"GBIF download failed for {species_name}: {status}")
            time.sleep(5)

        ### download + unzip
        zip_path = occ.download_get(download_key, path=gbif_dir)["path"]
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(gbif_dir)

        csv_files = glob(os.path.join(gbif_dir, "*.csv"))

    ### get the path
    gbif_path = csv_files[0]

    ### read the csv
    gbif_df = pd.read_csv(gbif_path, delimiter="\t", low_memory=False)

    ### return the results
    return gbif_df, gbif_path, species_key

### Run the download function

In [10]:
### create a dictionary for the species we want to download, and the dir to put it in
species_dirs = {
    "Antilocapra americana": gbif_pronghorn_dir,
    "Cynomys ludovicianus": gbif_prairie_dog_dir
}

### create an empty dictionary for the download
gbif_data = {}

### loop through the download function with our dictionary
for species_name, species_dir in species_dirs.items():
    gbif_df, gbif_path, species_key = download_gbif_species(
        species_name,
        species_dir
    )

    gbif_data[species_name] = {
        "df": gbif_df,
        "path": gbif_path,
        "species_key": species_key
    }

    ### print the results of the download
    print(f"\nLoaded {species_name}")
    print(f"Path: {gbif_path}")
    print(gbif_df.head())

Antilocapra americana: 2440902

Loaded Antilocapra americana
Path: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\gbif\gbif_pronghorn\0023629-260409193756587.csv
      gbifID                            datasetKey  \
0  930740086  0096dfc0-9925-47ef-9700-9b77814295f1   
1  923922129  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
2  923922121  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  923920930  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
4  923918563  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                                        occurrenceID   kingdom    phylum  \
0  http://bioimages.vanderbilt.edu/ind-baskauf/14...  Animalia  Chordata   
1     http://www.inaturalist.org/observations/752321  Animalia  Chordata   
2     http://www.inaturalist.org/observations/752191  Animalia  Chordata   
3     http://www.inaturalist.org/observations/749240  Animalia  Chordata   
4     http://www.inaturalist.org/observations/743215  Animalia  Chordata   

      class         order          famil

In [11]:
### save them as dataframes
gbif_pronghorn_df = gbif_data["Antilocapra americana"]["df"]
gbif_prairie_dog_df = gbif_data["Cynomys ludovicianus"]["df"]

In [12]:
### Make these spatial data frames (geodataframes)
for species_name, species_data in gbif_data.items():
    gbif_df = species_data["df"].dropna(
        subset=["decimalLongitude", "decimalLatitude"]
    ).copy()

    gbif_gdf = gpd.GeoDataFrame(
        gbif_df,
        geometry=gpd.points_from_xy(
            gbif_df["decimalLongitude"],
            gbif_df["decimalLatitude"]
        ),
        crs="EPSG:4326"
    )

    gbif_data[species_name]["gdf"] = gbif_gdf

    print(f"\nCreated GeoDataFrame for {species_name}")
    print(gbif_gdf.head())


Created GeoDataFrame for Antilocapra americana
      gbifID                            datasetKey  \
0  930740086  0096dfc0-9925-47ef-9700-9b77814295f1   
1  923922129  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
2  923922121  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  923920930  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
4  923918563  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                                        occurrenceID   kingdom    phylum  \
0  http://bioimages.vanderbilt.edu/ind-baskauf/14...  Animalia  Chordata   
1     http://www.inaturalist.org/observations/752321  Animalia  Chordata   
2     http://www.inaturalist.org/observations/752191  Animalia  Chordata   
3     http://www.inaturalist.org/observations/749240  Animalia  Chordata   
4     http://www.inaturalist.org/observations/743215  Animalia  Chordata   

      class         order          family        genus                species  \
0  Mammalia  Artiodactyla  Antilocapridae  Antilocapra  Antilocapra americana   
1  Mamma

In [13]:
### save the geodataframes
pronghorn_gdf = gbif_data["Antilocapra americana"]["gdf"]
prairie_dog_gdf = gbif_data["Cynomys ludovicianus"]["gdf"]

In [14]:
### combine into one gdf
gbif_mammals_gdf = pd.concat(
    [pronghorn_gdf, prairie_dog_gdf],
    ignore_index=True
)

In [15]:
### make a preliminary plot of the data
gbif_mammals_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Prairie Dog and Pronghorn Occurrences in GBIF",
    frame_width=600,
    size=40,
    cmap={"Cynomys ludovicianus": "yellow", "Antilocapra americana": "red"}
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]   (species)

In [16]:
### make a preliminary plot of the data with the Western Pawnee boundary
points = gbif_mammals_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Prairie Dog and Pronghorn Occurrences in GBIF",
    frame_width=600,
    size=40
)

boundary = master_bound_west_gdf.hvplot(
    geo=True,
    color=None,
    line_color="yellow",
    line_width=2
)

points * boundary

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (species)
   .Polygons.I :Polygons   [Longitude,Latitude]

In [17]:
### clip in the original shared CRS
gbif_mammals_clipped = gpd.clip(gbif_mammals_gdf, master_bound_west_gdf)

In [18]:
### save it to its directory
gbif_clipped_path = os.path.join(gbif_clipped_dir, "gbif_mammals_clipped.shp")
gbif_mammals_clipped.to_file(gbif_clipped_path)

C:\Users\warno\AppData\Local\Temp\ipykernel_20812\4207917777.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gbif_mammals_clipped.to_file(gbif_clipped_path)
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'occurrenceID' to 'occurrence'
  ogr_write(
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'infraspecificEpithet' to 'infraspeci'
  ogr_write(
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'scientificName' to 'scientific'
  ogr_write(
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'verbatimScientificName' to 'verbatimSc'
  ogr_write(
c:\Users\warno\minicond

In [23]:
### plot the result
points = gbif_mammals_clipped.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Clipped GBIF Mammal Occurrences (Western Pawnee)",
    frame_width=600,
    frame_height=500,
    size=40,
    cmap={"Cynomys ludovicianus": "yellow", "Antilocapra americana": "red"}
)

boundary = master_bound_west_gdf.hvplot(
    geo=True,
    fill_alpha=0,
    line_color="yellow",
    line_width=2
)

gbif_animals_plot = points * boundary
gbif_animals_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (species)
   .Polygons.I :Polygons   [Longitude,Latitude]

### Universal call for GBIF Mammal GDF

In [20]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main GBIF dir
gbif_dir = os.path.join(data_dir, 'gbif')

### GBIF clipped dir
gbif_clipped_dir = os.path.join(gbif_dir, 'gbif_clipped')

### GBIF mammal shapefile
gbif_mammal_path = os.path.join(gbif_clipped_dir, "gbif_mammals_clipped.shp")

### GBIF mammal convert to gdf
gbif_mammal_gdf = gpd.read_file(gbif_mammal_path)

In [25]:
### save an interactive html version of the plot
clipped_animals_plot_path = os.path.join(gbif_animal_fig_dir, 'gbif_animals_clipped_map.html')
hv.save(gbif_animals_plot, clipped_animals_plot_path)

print(f'Saved clipped GBIF map to: {clipped_animals_plot_path}')

Saved clipped GBIF map to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\figures\animals\gbif_animals_clipped_map.html
